# Config.py Reference

Every JAX-ALFA simulation is controlled by a single file: **`Config.py`**, located in the run directory.  
The file is plain Python — assign values to the variables listed below, and JAX-ALFA picks them up automatically.

```
export JAXALFA_RUNDIR=/path/to/run_directory
python -m src.Main
```

Parameters are grouped below in the same order they appear in `Config.py`.

---
## 1. Platform

| Parameter | Values | Default | Description |
|---|---|---|---|
| `use_double_precision` | `True` / `False` | `False` | Enable 64-bit (double) floating-point arithmetic throughout |
| `optGPU` | `0` / `1` | — | `0` = CPU only; `1` = use GPU |
| `GPU_ID` | integer ≥ 0 | `0` | Which GPU to use on a multi-GPU machine (0-indexed) |

**Recommendations**
- Use `use_double_precision = True` for production runs and benchmarking.  
  Single precision (`False`) can be used for exploratory runs and is ~2× faster on consumer GPUs,
  but may accumulate rounding error in long integrations or fine grids.
- On a SLURM cluster, `CUDA_VISIBLE_DEVICES` is set by the scheduler; `GPU_ID` is only used for interactive/local runs.

```python
# Recommended production settings
use_double_precision = True
optGPU = 1
GPU_ID = 0
```

---
## 2. Domain

| Parameter | Unit | Description |
|---|---|---|
| `l_x` | m | Domain length in x (streamwise) |
| `l_y` | m | Domain length in y (spanwise) |
| `l_z` | m | Domain height in z |
| `nx` | — | Number of grid points in x |
| `ny` | — | Number of grid points in y |
| `nz` | — | Number of grid points in z |

**Recommendations**
- Keep `nx = ny` and use powers of 2 (64, 128, 256, 384, 512) for maximum FFT efficiency.  
  Non-powers-of-2 work but are slower.
- The horizontal grid spacing is `dx = l_x / nx`.  
  For standard ABL simulations, aim for `dx ≈ dy ≈ dz` (isotropic grid).
- `l_z` should comfortably contain the full boundary-layer depth plus a damping layer above.

```python
# Example: 5 km × 5 km × 2 km domain at 128³
l_x, l_y, l_z = 5000, 5000, 2000
nx, ny, nz    = 128, 128, 128
```

---
## 3. Time Integration

| Parameter | Unit | Default | Description                                                                       |
|---|---|---|-----------------------------------------------------------------------------------|
| `istep` | — | `1` | Starting step number; set > 1 only for restart runs (not yet implemented)         |
| `dt` | s | — | Time-step size                                                                    |
| `SimTime` | s | — | Total physical simulation time                                                    |
| `Ugal` | m/s | — | Galilean transformation velocity (subtracted from the mean wind before advancing) |

**Recommendations**
- Choose `dt` so the CFL number `U_max * dt / dx < 0.2 (conservative).
- Set `Ugal` to the mean wind speed (e.g., `Ug2`) to reduce the advective CFL and allow a larger `dt`.
- For a restart run, set `istep` to the last saved step number from the previous run.

```python
istep   = 1          # fresh start
dt      = 0.5        # s — adjust for your CFL
SimTime = 36000      # s (10 hours)
Ugal    = 8          # m/s — match approximate mean wind
```

---
## 4. Surface Boundary Conditions

| Parameter | Values / Unit | Default | Description                                                                                             |
|---|---|---|---------------------------------------------------------------------------------------------------------|
| `optSurfFlux` | `0` / `1` | — | `0` = horizontally homogeneous boundary condition; `1` = heterogeneous (spatially varying)              |
| `optSurfBC` | `0` / `1` / `2` | `0` | `0` = constant heat flux; `1` = time-varying flux (file); `2` = time-varying surface temperature (file) |
| `z0m` | m | — | Aerodynamic roughness length for momentum                                                               |
| `z0T` | m | — | Scalar (thermal) roughness length                                                                       |
| `zTemperature` | m | `0.0` | Screen-height for heat-flux denominator; `0` = use `z0T`                                                |
| `SensibleHeatFlux` | K m/s | — | Constant surface heat flux; used only when `optSurfBC = 0`                                              |
| `SurfaceBCFile` | path | `'input/SurfaceBC.npz'` | Path to time-varying surface BC file (relative to run directory)                                        |

**Recommendations**
- For most idealized cases, `optSurfFlux = 0` and `optSurfBC = 0` (constant, homogeneous flux).
- For real-case simulations (e.g., GABLS3, Wangara), use `optSurfBC = 2` with prescribed surface temperatures.
- `zTemperature > 0` is needed when the observational temperature is at a screen height (e.g., 1.2 m for Wangara, 0.25 m for GABLS3) rather than at `z0T`.
- Positive `SensibleHeatFlux` = upward heat flux (unstable/convective); negative = downward (stable).

```python
# Idealized stable BL (constant cooling)
optSurfFlux      = 0
optSurfBC        = 0
z0m, z0T         = 0.1, 0.1
SensibleHeatFlux = -0.03    # K m/s downward

# Real-case with observed surface temperatures (GABLS3-style)
optSurfBC        = 2
zTemperature     = 0.25     # m
SurfaceBCFile    = 'input/SurfaceBC.npz'
```

---
## 5. Forcing

| Parameter | Values / Unit | Default | Description |
|---|---|---|---|
| `optGeoWind` | `0` / `1` | `0` | `0` = constant `Ug2`, `Vg2`; `1` = time + height varying (from file) |
| `Ug2` | m/s | — | Geostrophic wind component in x |
| `Vg2` | m/s | — | Geostrophic wind component in y |
| `GeoWindFile` | path | `'input/GeoWind.npz'` | Path to time/height-varying geostrophic wind file |
| `f_coriolis` | 1/s | — | Coriolis parameter (typically `1e-4` at mid-latitudes) |
| `inversion` | K/m | — | Potential temperature lapse rate above domain top (free-atmosphere stratification) |
| `optBuoyancy` | `0` / `1` | — | `0` = buoyancy uses fixed reference temperature `T_0`; `1` = uses local virtual potential temperature |
| `T_0` | K | — | Reference temperature for buoyancy (used when `optBuoyancy = 0`, and always for non-dimensionalisation) |

**Recommendations**
- Use `optBuoyancy = 1` for all simulations. `optBuoyancy = 0` is rarely used.
- Set `inversion > 0` to cap convective boundary layers.

```python
optGeoWind  = 0
Ug2, Vg2    = 8.0, 0.0     # m/s
f_coriolis  = 1.0e-4       # 1/s
inversion   = 3.0 / 1000   # 3 K/km
optBuoyancy = 1
T_0         = 300.0        # K
```

---
## 6. Subgrid-Scale (SGS) Model

| Parameter | Values | Default | Description |
|---|---|---|---|
| `optSgs` | `1` – `4` | — | SGS model selection (see table below) |
| `dynamicSGS_call_time` | integer ≥ 1 | — | Recompute dynamic SGS coefficients every N steps |
| `FGR` | `1` or integer ≥ 2 | — | Filter-to-grid ratio: `1` = implicit filtering + dealiasing (standard); `≥ 2` = explicit filtering, no dealiasing |
| `Cs2` | float | — | Initial `Cs²` for SM models (used before first dynamic update) |
| `Cwl` | float | — | Initial `C_WL` for WL models (used before first dynamic update) |
| `Cs2PrRatio` | float | — | Initial `Cs² / Prt` for SM models |
| `CwlPrRatio` | float | — | Initial `C_WL / Prt` for WL models |

### SGS model options

| `optSgs` | Model | Best for |
|---|---|---|
| `1` | LASDD-SM (Locally-Averaged Scale-Dependent Dynamic Smagorinsky) | General ABL; most thoroughly validated |
| `2` | LASDD-WL (Wong–Lilly variant) | Alternative to LASDD-SM |
| `3` | LAD-SM (Locally-Averaged Dynamic Smagorinsky) | Simpler dynamic procedure; faster per step |
| `4` | LAD-WL | Wong–Lilly variant of LAD |

**Recommendations**
- `optSgs = 1` (LASDD-SM) is the recommended default for all stable and neutral BL simulations.
- `FGR = 1` (implicit filtering) is standard. Dealiasing is activated automatically.
  Use `FGR = 2` only if you need explicit filtering (dealiasing is not needed).
- `dynamicSGS_call_time = 1` (every step) is standard. Increase to 5–10 for faster runs where the SGS field evolves slowly.
- The initial values `Cs2 = 0.01` and `Cwl = 0.01` (`0.1²`) are reasonable warm-start guesses; they are replaced after the first dynamic update.

```python
optSgs              = 1          # LASDD-SM
dynamicSGS_call_time = 1
FGR                 = 1
Cs2                 = 0.1 ** 2
Cwl                 = 0.1 ** 2
Cs2PrRatio          = Cs2 / 1.0
CwlPrRatio          = Cwl / 1.0
```

---
## 7. Rayleigh Damping Layer

| Parameter | Values / Unit | Default | Description |
|---|---|---|---|
| `optDamping` | `0` / `1` | — | `1` = activate Rayleigh damping above `z_damping` |
| `z_damping` | m | — | Height above which damping starts |
| `RelaxTime` | s | — | Damping relaxation time scale (larger = weaker damping) |

**Recommendations**
- Always enable the damping layer (`optDamping = 1`) to absorb upward-propagating disturbances and prevent artificial reflections from the rigid lid.
- Set `z_damping` to roughly the top of the boundary layer (or ~60–70% of `l_z`), leaving at least 20–30% of the domain depth as the absorbing sponge.

```python
optDamping = 1
z_damping  = 1500    # m — top of the ABL
RelaxTime  = 300     # s
```

---
## 8. Statistics and Output

| Parameter | Unit | Description |
|---|---|---|
| `SampleInterval_sec` | s | Collect one instantaneous horizontal-average sample every N seconds |
| `OutputInterval_sec` | s | Write time-averaged statistics to disk every N seconds |
| `Output3DInterval_sec` | s | Write full 3-D field snapshots every N seconds |

**Recommendations**
- `OutputInterval_sec` should be a multiple of `SampleInterval_sec`; averages accumulate from individual samples.
- 3-D field output is expensive in both compute and storage. For long runs, set `Output3DInterval_sec = SimTime` to write only at the end, or to a multiple of the boundary-layer turnover time (~`l_z / u_star`).
- Typical values for a 10-hour ABL run: `SampleInterval_sec = 15`, `OutputInterval_sec = 300`.

```python
SampleInterval_sec   = 15.0        # s
OutputInterval_sec   = 300.0       # s
Output3DInterval_sec = SimTime     # write once at end
```

---
## 9. Large-Scale Advection Forcing

| Parameter | Values | Default | Description |
|---|---|---|---|
| `optAdvection` | `0` / `1` | `0` | `0` = no mesoscale advection; `1` = time + height varying (from file) |
| `AdvectionFile` | path | `'input/AdvForcing.npz'` | Path to advection forcing file |

**Recommendations**
- Most idealized cases need no advection forcing (`optAdvection = 0`).
- Enable for real-case simulations where mesoscale horizontal advection of temperature or moisture is significant (e.g., GABLS3, Wangara diurnal cycle).

```python
optAdvection = 0               # most idealized cases

optAdvection  = 1              # real-case with advection
AdvectionFile = 'input/AdvForcing.npz'
```

---
## 10. Moisture

| Parameter | Values / Unit | Default | Description |
|---|---|---|---|
| `optMoisture` | `0` / `1` | `0` | `0` = dry run; `1` = prognostic specific humidity Q |
| `zMoisture` | m | `0.0` | Screen height for moisture-flux denominator; `0` = use `z0T` |
| `MoistureFlux` | kg/kg m/s | `0.0` | Constant surface moisture flux; used when `optMoistureSurfBC = 0` |
| `optMoistureSurfBC` | `0` / `1` / `2` | `0` | `0` = constant flux; `1` = time-varying flux (file); `2` = time-varying surface Q (file) |
| `MoistureSurfaceBCFile` | path | `'input/MoistureSurfaceBC.npz'` | Path to time-varying moisture BC file |
| `q_inversion` | kg/kg/m | `0.0` | Specific humidity lapse rate above domain top; `0` = zero gradient |

**Recommendations**
- Most ABL benchmarks (GABLS1, GABLS2, NBL_A94) are dry; leave `optMoisture = 0`.
- Enable moisture for the diurnal cycle (Wangara) or when dew formation matters (GABLS3).
- `zMoisture` follows the same logic as `zTemperature`: set to the screen height of the moisture observation when `optMoistureSurfBC = 2`.

```python
# Dry run (default for most idealized cases)
optMoisture = 0

# Moist real-case
optMoisture          = 1
zMoisture            = 0.25
optMoistureSurfBC    = 2
MoistureSurfaceBCFile = 'input/MoistureSurfaceBC.npz'
```

---
## 11. Pressure Solver

| Parameter | Values | Default | Description |
|---|---|---|---|
| `optPressureSolver` | `0` / `1` | `0` | `0` = LU (dense matrix); `1` = Thomas tridiagonal algorithm (recommended) |

**Recommendations**
- Always use `optPressureSolver = 1` (Thomas algorithm). It is ~2× faster overall on a 128³ double-precision run because it exploits the tridiagonal structure of the pressure Poisson system instead of applying a dense O(N³) LU solve per horizontal wavenumber pair.
- The Thomas solver produces bit-for-bit identical results to the LU solver for all non-singular modes.
- `optPressureSolver = 0` is retained only for regression testing and backward compatibility.

```python
optPressureSolver = 1    # Thomas — use this in all production runs
```

---
## Quick-Reference Summary

The table below shows the recommended values for a typical production run.

| Parameter | Recommended | Note |
|---|---|---|
| `use_double_precision` | `True` | Higher accuracy; slower on consumer GPUs |
| `optGPU` | `1` | Use GPU |
| `optSurfFlux` | `0` | Homogeneous surface for idealized runs |
| `optSurfBC` | `0` or `2` | `0` for constant flux; `2` for observed surface temperature |
| `optBuoyancy` | `1` | Virtual potential temperature buoyancy |
| `optSgs` | `1` | LASDD-SM — best validated SGS model |
| `FGR` | `1` | Implicit filtering |
| `dynamicSGS_call_time` | `1` | Every step for accuracy |
| `optDamping` | `1` | Always enable the sponge layer |
| `optAdvection` | `0` | Unless real-case forcing is needed |
| `optMoisture` | `0` | Dry unless moisture is a key variable |
| `optPressureSolver` | `1` | Thomas algorithm — 2× faster |

---
*Generated for JAX-ALFA — see the [GitHub repository](https://github.com/Sukantabasu/jax-alfa) for source code and issue tracking.*